# Micro-NAS: Comparação das 5 SSt mais utilizadas na literatura

Este notebook implementa e compara, em ambiente controlado, as cinco Estratégias de Busca (SSt) mais recorrentes na literatura de Neural Architecture Search:

1. **Random Search (RS)**
2. **Evolutionary Algorithm (EA)**
3. **Reinforcement Learning (RL)**
4. **Bayesian Optimization (BO)**
5. **Gradient-based / Hill Climbing (GO)**

Para garantir uma comparação justa, cada SSt recebe exatamente **10 avaliações reais** (chamadas à VSt). O objetivo é observar qual estratégia extrai o melhor resultado dado o mesmo orçamento computacional — comportamento análogo ao que ocorre em experimentos NAS de baixa fidelidade na literatura.

## 1. Definição do SSp e da VSt

### Search Space (SSp)

O dicionário `search_space` define o conjunto discreto de hiperparâmetros que o algoritmo NAS pode combinar para montar uma arquitetura candidata. Trata-se de um SSp do tipo *layer-based*: a rede sempre segue a mesma topologia sequencial, e a busca se concentra nos valores de cada hiperparâmetro por camada.

### Validation Strategy (VSt)

A função `avaliar_arquitetura` implementa a VSt do tipo *Partial Training*: cada candidato é treinado por apenas 1 época em um subconjunto reduzido do MNIST (3.000 amostras de treino, 1.000 de validação). Essa abordagem de baixa fidelidade (*low-fidelity proxy*) reduz drasticamente o custo computacional de cada avaliação, permitindo comparar as SSt dentro de um tempo viável.

A função retorna a acurácia de validação, que é o sinal de desempenho repassado a cada SSt.

In [ ]:
import numpy as np
import random
import time
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from sklearn.ensemble import RandomForestRegressor
import logging

tf.get_logger().setLevel(logging.ERROR)

# --- Dados (VSt de baixa fidelidade: subset pequeno do MNIST) ---
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train[:3000].astype("float32") / 255.0
y_train = y_train[:3000]
x_test  = x_test[:1000].astype("float32") / 255.0
y_test  = y_test[:1000]
x_train = np.expand_dims(x_train, -1)
x_test  = np.expand_dims(x_test,  -1)

# --- SSp: espaço de busca discreto (layer-based) ---
search_space = {
    'filters':      [16, 32, 64],
    'kernel_size':  [3, 5],
    'activation':   ['relu', 'elu', 'tanh'],
    'dropout':      [0.1, 0.3, 0.5]
}

# --- VSt: treina 1 época e retorna acurácia de validação ---
def avaliar_arquitetura(params):
    model = models.Sequential([
        Input(shape=(28, 28, 1)),
        layers.Conv2D(params['filters'], params['kernel_size'], activation=params['activation']),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dropout(params['dropout']),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(x_train, y_train, epochs=1, validation_data=(x_test, y_test), verbose=0)
    return history.history['val_accuracy'][0]

# --- Utilitário: amostra aleatória uniforme do SSp ---
def gerar_aleatorio():
    return {k: random.choice(v) for k, v in search_space.items()}

## 2. Implementação das SSt

Cada função abaixo recebe `max_evals` (orçamento de avaliações reais) e retorna a melhor arquitetura encontrada junto com sua acurácia de validação. As implementações são intencionalmente simplificadas para expor o mecanismo central de cada SSt, sem dependências externas especializadas.

### 2.1 Random Search (RS)
Amostra arquiteturas de forma independente e uniforme do SSp a cada iteração. Não usa nenhuma informação de avaliações anteriores. É o baseline mais simples e, por isso, o ponto de referência para avaliar se uma SSt mais sofisticada realmente agrega valor dado o mesmo orçamento.

### 2.2 Evolutionary Algorithm (EA)
Mantém uma população de arquiteturas que evolui por operações de *crossover* (recombinação dos genes dos dois melhores indivíduos) e *mutação* (alteração aleatória de um hiperparâmetro com baixa probabilidade). Cada iteração gera um filho que substitui o pior indivíduo caso seja avaliado. Representa a classe de algoritmos baseados em princípios biológicos de seleção natural.

### 2.3 Reinforcement Learning (RL) — Multi-Armed Bandit
Modela cada hiperparâmetro como um *slot machine* independente (bandit). O agente mantém uma distribuição de probabilidades sobre os valores de cada hiperparâmetro e a atualiza com base na recompensa obtida (acurácia): escolhas que levaram a bons resultados recebem reforço positivo (probabilidade aumentada), enquanto as demais ficam relativamente menos prováveis. O sinal de reforço é calculado **antes** de atualizar o registro do melhor resultado, garantindo que a comparação seja feita contra o melhor histórico anterior.

### 2.4 Bayesian Optimization (BO)
Usa um modelo substituto (*surrogate model*) — aqui implementado como um Random Forest — para estimar a acurácia esperada de qualquer ponto do SSp sem precisar avaliá-lo na VSt. Em cada iteração: (1) treina o surrogate com o histórico de avaliações; (2) gera um conjunto de candidatos aleatórios; (3) escolhe o candidato com maior acurácia prevista pelo surrogate para avaliação real. As primeiras 3 avaliações são aleatórias (*warm-up*) para inicializar o modelo substituto.

### 2.5 Hill Climbing / Gradient-based (GO)
Análogo discreto da descida de gradiente. Parte de uma arquitetura inicial e, a cada passo, gera um vizinho alterando exatamente um hiperparâmetro. Move-se para o vizinho somente se ele for melhor que o estado atual — comportamento equivalente a seguir o gradiente positivo no espaço contínuo. É susceptível a ótimos locais, pois nunca aceita movimentos que piorem a acurácia.

In [ ]:
# --- 1. Random Search ---
def random_search(max_evals):
    melhor_acc, melhor_arq = 0, None
    for _ in range(max_evals):
        arq = gerar_aleatorio()
        acc = avaliar_arquitetura(arq)
        if acc > melhor_acc:
            melhor_acc, melhor_arq = acc, arq
    return melhor_arq, melhor_acc


# --- 2. Evolutionary Algorithm ---
def evolutionary_search(max_evals):
    # Inicializa a população: cada indivíduo tem arquitetura e acurácia correspondente
    populacao = []
    for _ in range(2):
        arq = gerar_aleatorio()
        acc = avaliar_arquitetura(arq)   # avalia a MESMA arquitetura gerada
        populacao.append({'arq': arq, 'acc': acc})
    evals = 2

    while evals < max_evals:
        # Seleciona os dois melhores como pais
        populacao.sort(key=lambda x: x['acc'], reverse=True)
        pai, mae = populacao[0], populacao[1]

        # Crossover uniforme: cada gene vem do pai ou da mãe com prob. 0.5
        filho_arq = {
            k: pai['arq'][k] if random.random() > 0.5 else mae['arq'][k]
            for k in search_space
        }

        # Mutação: com prob. 0.3, um gene aleatório é substituído por valor aleatório
        if random.random() < 0.3:
            gene = random.choice(list(search_space.keys()))
            filho_arq[gene] = random.choice(search_space[gene])

        acc = avaliar_arquitetura(filho_arq)
        populacao.append({'arq': filho_arq, 'acc': acc})
        evals += 1

    populacao.sort(key=lambda x: x['acc'], reverse=True)
    return populacao[0]['arq'], populacao[0]['acc']


# --- 3. Reinforcement Learning (Multi-Armed Bandit por hiperparâmetro) ---
def rl_search(max_evals):
    # Distribuição inicial uniforme sobre os valores de cada hiperparâmetro
    probs = {
        k: {v: 1.0 / len(search_space[k]) for v in search_space[k]}
        for k in search_space
    }
    melhor_acc, melhor_arq = 0, None

    for _ in range(max_evals):
        # Amostra uma arquitetura seguindo a distribuição atual
        arq = {
            k: random.choices(list(probs[k].keys()), weights=list(probs[k].values()), k=1)[0]
            for k in search_space
        }

        acc = avaliar_arquitetura(arq)

        # Reforço positivo: aplica ANTES de atualizar melhor_acc para comparar
        # com o melhor resultado anterior (não com o atual)
        if acc >= melhor_acc * 0.95:
            for k, v in arq.items():
                probs[k][v] *= 1.2  # aumenta a probabilidade dos genes bem-sucedidos

        if acc > melhor_acc:
            melhor_acc, melhor_arq = acc, arq

    return melhor_arq, melhor_acc


# --- 4. Bayesian Optimization (surrogate via Random Forest) ---
def bayesian_search(max_evals):
    # Codifica a arquitetura como vetor de índices inteiros para o surrogate
    def dict_to_vec(arq):
        return [search_space[k].index(arq[k]) for k in search_space]

    historico_X, historico_y = [], []
    melhor_acc, melhor_arq = 0, None
    surrogate = RandomForestRegressor(n_estimators=10, random_state=42)

    # Warm-up: avaliações aleatórias para inicializar o modelo substituto
    n_warmup = min(3, max_evals)
    for _ in range(n_warmup):
        arq = gerar_aleatorio()
        acc = avaliar_arquitetura(arq)
        historico_X.append(dict_to_vec(arq))
        historico_y.append(acc)
        if acc > melhor_acc:
            melhor_acc, melhor_arq = acc, arq

    # Loop principal: surrogate orienta qual candidato avaliar
    for _ in range(n_warmup, max_evals):
        surrogate.fit(historico_X, historico_y)

        # Gera pool de candidatos e seleciona o de maior acurácia prevista
        candidatos = [gerar_aleatorio() for _ in range(50)]
        previsoes  = surrogate.predict([dict_to_vec(c) for c in candidatos])
        melhor_candidato = candidatos[np.argmax(previsoes)]

        acc = avaliar_arquitetura(melhor_candidato)
        historico_X.append(dict_to_vec(melhor_candidato))
        historico_y.append(acc)
        if acc > melhor_acc:
            melhor_acc, melhor_arq = acc, melhor_candidato

    return melhor_arq, melhor_acc


# --- 5. Hill Climbing (análogo discreto do gradient-based) ---
def hill_climbing_search(max_evals):
    # Ponto inicial aleatório no SSp
    atual_arq = gerar_aleatorio()
    atual_acc = avaliar_arquitetura(atual_arq)
    evals = 1

    while evals < max_evals:
        # Gera vizinho: altera exatamente um hiperparâmetro (um passo no SSp)
        vizinho_arq = atual_arq.copy()
        param = random.choice(list(search_space.keys()))
        vizinho_arq[param] = random.choice(search_space[param])

        vizinho_acc = avaliar_arquitetura(vizinho_arq)
        evals += 1

        # Aceita o vizinho somente se melhorar (greedy / gradiente positivo)
        if vizinho_acc > atual_acc:
            atual_arq, atual_acc = vizinho_arq, vizinho_acc

    return atual_arq, atual_acc

## 3. Execução comparativa

Cada SSt é executada com o mesmo orçamento de `max_avaliacoes` chamadas reais à VSt. O tempo de execução e a melhor acurácia de validação encontrada são registrados para compor o quadro comparativo final.

Como todas as avaliações usam a mesma VSt de baixa fidelidade (1 época, subset reduzido), diferenças de acurácia refletem a capacidade de cada SSt em explorar o SSp de forma eficiente — não diferenças de poder computacional ou tempo de treinamento.

In [ ]:
max_avaliacoes = 10
resultados = {}

otimizadores = {
    "Random Search (RS)": random_search,
    "Evolutionary (EA) ": evolutionary_search,
    "Reinforcement (RL) ": rl_search,
    "Bayesian (BO)      ": bayesian_search,
    "Hill Climbing (GO) ": hill_climbing_search,
}

print(f"Orçamento por SSt: {max_avaliacoes} avaliações reais (VSt: 1 época, MNIST 3k/1k)")
print("-" * 60)

for nome, funcao in otimizadores.items():
    inicio = time.time()
    melhor_arq, melhor_acc = funcao(max_avaliacoes)
    tempo = time.time() - inicio
    resultados[nome] = {"acc": melhor_acc, "tempo": tempo, "arq": melhor_arq}
    print(f"{nome} | Acc: {melhor_acc:.4f} | Tempo: {tempo:.1f}s")

print("\n" + "=" * 60)
print("RESULTADO FINAL (ordenado por acuracia de validacao)")
print("=" * 60)
for pos, (nome, res) in enumerate(
    sorted(resultados.items(), key=lambda x: x[1]['acc'], reverse=True), start=1
):
    print(f"{pos}. {nome} | Acc: {res['acc']:.4f} | Tempo: {res['tempo']:5.1f}s | Arq: {res['arq']}")